# GRU node-only v01 — shared selected features

Đối chứng không graph cho GAT-GRU. Mỗi service được xử lý độc lập bởi node MLP và GRU dùng chung trọng số. MLP khớp độ sâu/rộng GAT encoder; mô hình không nhận node khác hoặc edge features.

Quy trình: train trên train, chọn checkpoint và hyperparameters bằng validation, sau đó mới đánh giá 5 seed trên frozen test.

In [1]:
import sys
from pathlib import Path

import torch

ROOT = Path.cwd().resolve()
while not (ROOT / 'modeling-common').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'modeling-common'))
import forecast_ablation as fa

ctx = fa.load_context(ROOT)
selection = fa.load_selection_contract(
    ROOT / 'configs' / 'gatgru' / 'feature_contract_v01.json',
    require_edge=False,
)
base_cfg = fa.load_baseline_config(ctx)
base_cfg['use_edge_features'] = False
base_cfg['emb_dim'] = 4
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUT = (ROOT / 'artifacts' / 'gru' /
       'v01_shared_selected_features')

print('node features:', selection['node_features'])
print('edge features: []')
print('model kind: gru_node_only')
print('device:', DEVICE)
print('output:', OUT)

node features: ['rps_in', 'rps_in_delta', 'rps_per_replica', 'latency_p50_ms', 'latency_p95_ms', 'cpu_cores', 'mem_mib', 'net_rx_kBps', 'net_tx_kBps', 'replicas']
edge features: []
model kind: gru_node_only
device: cpu
output: F:\0. UIT\KLTN\thesis-gnn-rl-autoscaling\lstm-load-testing\experiments\gru\v01_shared_selected_features


## 1. Tune training hyperparameters trên validation

Node MLP, GRU hidden size, feature contract và embedding được khóa. Test chưa được truy cập ở bước này.

In [2]:
best_cfg, study = fa.run_optuna_fixed_architecture(
    ctx,
    'gru_node_only',
    selection['node_features'],
    [],
    base_cfg,
    OUT / 'tuning',
    n_trials=24,
    seed=42,
    device=DEVICE,
    score_weights=fa.CPU_DIAGNOSTIC_WEIGHTS,
    parent_version='gatgru_v05_selected_features_final',
)
assert best_cfg['use_edge_features'] is False
print('best validation score:', study.best_value)
best_cfg

[I 2026-10-24 09:53:37,889] A new study created in RDB with name: gru_node_only_b55e5fc117df47d5_fixed_arch
[I 2026-10-24 10:05:15,390] Trial 0 finished with value: 0.1108176247476433 and parameters: {'window': 18, 'dropout': 0.11973169683940732, 'lr': 0.0006427207987637223, 'weight_decay': 2.9375384576328313e-06, 'batch_size': 64}. Best is trial 0 with value: 0.1108176247476433.
[I 2026-10-24 10:22:09,818] Trial 1 finished with value: 0.11077626963396761 and parameters: {'window': 24, 'dropout': 0.16648852816008436, 'lr': 0.0007037018186923737, 'weight_decay': 3.5113563139704077e-06, 'batch_size': 128}. Best is trial 1 with value: 0.11077626963396761.
[I 2026-10-24 10:30:32,015] Trial 2 finished with value: 0.11007516081103624 and parameters: {'window': 24, 'dropout': 0.027898772130408367, 'lr': 0.0008001480489146253, 'weight_decay': 1.2562773503807034e-05, 'batch_size': 64}. Best is trial 2 with value: 0.11007516081103624.
[I 2026-10-24 10:39:18,209] Trial 3 finished with value: 0.11

best validation score: 0.10893487266625594


{'window': 18,
 'dropout': 0.012531365296947577,
 'lr': 0.0011888978467149338,
 'batch_size': 128,
 'gat_hidden': 32,
 'heads': 4,
 'gat_layers': 2,
 'gru_hidden': 128,
 'edge_direction': 'both',
 'use_edge_features': False,
 'emb_dim': 4,
 'weight_decay': 3.668595806394715e-05}

## 2. Huấn luyện 5 seed và đánh giá frozen test

Mỗi seed huấn luyện trên train và dùng validation để chọn checkpoint. `run_final` không refit trên train + validation.

In [3]:
prediction_path, training_meta = fa.run_final(
    ctx,
    'gru_node_only',
    selection['node_features'],
    [],
    best_cfg,
    OUT,
    seeds=(0, 1, 2, 3, 4),
    device=DEVICE,
    model_name='gru_node_only',
    parent_version='gatgru_v05_selected_features_final',
)
assert training_meta['model_kind'] == 'gru_node_only'
print(prediction_path)
training_meta

F:\0. UIT\KLTN\thesis-gnn-rl-autoscaling\lstm-load-testing\experiments\gru\v01_shared_selected_features\gru_node_only_predictions_test.npz


{'model': 'gru_node_only',
 'model_kind': 'gru_node_only',
 'config': {'window': 18,
  'dropout': 0.012531365296947577,
  'lr': 0.0011888978467149338,
  'batch_size': 128,
  'gat_hidden': 32,
  'heads': 4,
  'gat_layers': 2,
  'gru_hidden': 128,
  'edge_direction': 'both',
  'use_edge_features': False,
  'emb_dim': 4,
  'weight_decay': 3.668595806394715e-05},
 'seeds': [0, 1, 2, 3, 4],
 'val_mae_scaled': [0.11733268201351166,
  0.11861442774534225,
  0.11787841469049454,
  0.11640775948762894,
  0.11679934710264206],
 'node_features': ['rps_in',
  'rps_in_delta',
  'rps_per_replica',
  'latency_p50_ms',
  'latency_p95_ms',
  'cpu_cores',
  'mem_mib',
  'net_rx_kBps',
  'net_tx_kBps',
  'replicas'],
 'edge_features': [],
 'targets': ['rps_in', 'cpu_cores'],
 'services': ['frontend',
  'adservice',
  'cartservice',
  'checkoutservice',
  'currencyservice',
  'emailservice',
  'paymentservice',
  'productcatalogservice',
  'recommendationservice',
  'redis-cart',
  'shippingservice'],
 'n